# W12: 12주 성과 발표

도입 전/후 KPI를 막대그래프로 비교하고, Gemini로 경영진 보고서를 생성합니다.


In [ ]:
import io
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for font_path in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(font_path)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}

    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        out = model.generate_content(prompt)
        return getattr(out, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

kpi = pd.DataFrame({
    "KPI": ["매출", "주문건수", "재구매율", "클레임률"],
    "도입전": [8700000, 3100, 0.24, 0.05],
    "도입후": [10300000, 3650, 0.34, 0.03]
})

idx = np.arange(len(kpi))
width = 0.35
fig, ax = plt.subplots(figsize=(8,3))
ax.bar(idx - width/2, kpi["도입전"], width, label="도입전")
ax.bar(idx + width/2, kpi["도입후"], width, label="도입후")
ax.set_xticks(idx)
ax.set_xticklabels(kpi["KPI"], rotation=10)
ax.set_title("도입 전/후 KPI 비교")
ax.legend()
plt.tight_layout()
plt.show()

kpi["개선율"] = ((kpi["도입후"] - kpi["도입전"]) / kpi["도입전"] * 100).round(2)
print(kpi)

lines = [f"{r.KPI}: {r.도입전} → {r.도입후} ({r.개선율}%)" for r in kpi.itertuples()]
report = use_gemini(
    "아래 12주 성과 지표 기반으로 경영진용 보고서 작성(성과/위험/다음 액션):
" + "
".join(lines),
    "12주간 매출·주문·재구매율이 개선되었고 클레임률이 낮아졌습니다. 다음은 상위 상품 집중 운영과 A/B 테스트로 전환율을 더 높이겠습니다."
)
print(report)
